In [1]:
# ==================================================
# Project Path
# ==================================================

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]

sys.path.insert(
    0,
    str(PROJECT_ROOT)
)

In [2]:
# ==================================================
# Import Libraries
# ==================================================

from pathlib import Path
import sys

import pandas as pd
import mlflow
import mlflow.xgboost

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

from src.mlflow.tracking import MLflowTracker

d:\Subject\CV2026\Market Risk Classification\market-risk-classification\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ==================================================
# Project Path
# ==================================================

PROJECT_ROOT = Path.cwd().parents[1]

sys.path.insert(
    0,
    str(PROJECT_ROOT)
)

In [4]:
# ==================================================
# Load Data
# ==================================================

DATA_PATH = (
    r"D:\Subject\CV2026\Market Risk Classification"
    r"\market-risk-classification"
    r"\data\processed\BTCUSDT\BTCUSDT_1m_clean.csv"
)


df = pd.read_csv(
    DATA_PATH
)


df["open_time"] = pd.to_datetime(
    df["open_time"]
)


df = df.sort_values(
    "open_time"
)


df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume,ignore
0,2017-08-17 04:00:00,4261.48,4261.48,4261.48,4261.48,1.775183,2017-08-17 04:00:59.999,7564.906851,3,0.075183,320.390851,0
1,2017-08-17 04:01:00,4261.48,4261.48,4261.48,4261.48,0.000000,2017-08-17 04:01:59.999,0.000000,0,0.000000,0.000000,0
2,2017-08-17 04:02:00,4280.56,4280.56,4280.56,4280.56,0.261074,2017-08-17 04:02:59.999,1117.542921,2,0.261074,1117.542921,0
3,2017-08-17 04:03:00,4261.48,4261.48,4261.48,4261.48,0.012008,2017-08-17 04:03:59.999,51.171852,3,0.012008,51.171852,0
4,2017-08-17 04:04:00,4261.48,4261.48,4261.48,4261.48,0.140796,2017-08-17 04:04:59.999,599.999338,1,0.140796,599.999338,0


In [5]:
# ==================================================
# Create Target
# ==================================================

HORIZON = 10


df["target_10m"] = (

    df["close"].shift(-HORIZON)

    >

    df["close"]

).astype(int)


df = df.iloc[:-HORIZON]


df["target_10m"].value_counts()

target_10m
1    1008641
0     991349
Name: count, dtype: int64

In [6]:
# ==================================================
# Features
# ==================================================

FEATURES = [

    "open",

    "high",

    "low",

    "close",

    "volume",

    "quote_asset_volume",

    "number_of_trades",

    "taker_buy_base_volume",

    "taker_buy_quote_volume",

]


TARGET = "target_10m"


X = df[FEATURES]

y = df[TARGET]

In [7]:
# ==================================================
# Time Split
# ==================================================

split = int(
    len(df) * 0.8
)


X_train = X.iloc[:split]

X_test = X.iloc[split:]


y_train = y.iloc[:split]

y_test = y.iloc[split:]


print(
    X_test.shape
)

(399998, 9)


In [8]:
# ==================================================
# Load Model From MLflow
# ==================================================

from src.mlflow.tracking import MLflowTracker

# Khởi tạo để set đúng MLflow Tracking URI
tracker = MLflowTracker()


experiment = mlflow.get_experiment_by_name(
    "Market Risk Classification"
)


runs = mlflow.search_runs(

    experiment_ids=[
        experiment.experiment_id
    ]

)


runs[
[
    "run_id",
    "tags.mlflow.runName"
]
]

,run_id,tags.mlflow.runName
0,af98b0fb0b854f10a7e2779198b41cd8,Prediction_XGBoost_Return_10m
1,81b0cb8ed06141149b5a4fde13d04377,XGBoost_Return_30m
2,773c2125e2924bc6a9c974eb450d9d41,XGBoost_Return_15m
3,2c8d076419594c81adc3ab3fb87023a8,XGBoost_Return_10m
4,43e3443f1b08484b92fb6f2b82d96206,XGBoost_Return_5m
5,f09402f135bf49569fc47b8a44a0ac8a,XGBoost_Return_1m
6,76ebd23913a0421a82f8294da23f692f,XGBoost_Return_1m
7,cd33862de4bb4a9caa75a5b761a9f29a,XGBoost_Return_1m
8,dd20f85fe6da4466ae9207f0d140cbf9,XGBoost_Return_1m
9,5527a31df31a499582e883b422d24e42,Logistic Regression Feature Store


In [9]:
# ==================================================
# Select Return 10m Model
# ==================================================

run = runs[
    runs[
        "tags.mlflow.runName"
    ]
    ==
    "XGBoost_Return_10m"
]


run_id = run.iloc[0]["run_id"]


model_uri = f"runs:/{run_id}/model"


print(model_uri)

runs:/2c8d076419594c81adc3ab3fb87023a8/model


In [10]:
# ==================================================
# Load XGBoost Model
# ==================================================

model = mlflow.xgboost.load_model(
    model_uri
)


print(model)

XGBClassifier(base_score=[0.50441194], booster='gbtree', callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None,
              feature_types=['float', 'float', 'float', 'float', 'float',
                             'float', 'int', 'float', 'float'],
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)


In [11]:
# ==================================================
# Prediction
# ==================================================

y_pred = model.predict(
    X_test
)


y_prob = model.predict_proba(
    X_test
)[:,1]

In [12]:
# ==================================================
# Evaluation
# ==================================================

metrics = {

    "accuracy":
        accuracy_score(
            y_test,
            y_pred
        ),

    "precision":
        precision_score(
            y_test,
            y_pred
        ),

    "recall":
        recall_score(
            y_test,
            y_pred
        ),

    "f1":
        f1_score(
            y_test,
            y_pred
        ),

    "roc_auc":
        roc_auc_score(
            y_test,
            y_prob
        )

}


metrics

{'accuracy': 0.5009100045500228,
 'precision': 0.5039248797667133,
 'recall': 0.6231993333068773,
 'f1': 0.5572511804195618,
 'roc_auc': 0.5006126777207871}

In [13]:
# ==================================================
# Log Prediction Result
# ==================================================

tracker = MLflowTracker()


with tracker.start_run(

    run_name=f"Prediction_XGBoost_Return_{HORIZON}m"

):

    # ============================
    # Tags
    # ============================

    mlflow.set_tag(
        "model",
        "XGBoost"
    )


    mlflow.set_tag(
        "prediction_horizon",
        f"{HORIZON}m"
    )


    mlflow.set_tag(
        "task",
        "binary_classification"
    )


    # ============================
    # Params
    # ============================

    tracker.log_params({

        "model": "XGBoost",

        "prediction_type": "inference",

        "prediction_horizon": HORIZON

    })


    # ============================
    # Metrics
    # ============================

    tracker.log_metrics(
        metrics
    )


tracker.generate_summary()

MLflow training summary generated
Saved at: D:\Subject\CV2026\Market Risk Classification\market-risk-classification\artifacts\training_summary.txt
